In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor, ViTModel
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import itertools
import time
from PIL import Image


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [4]:
import os

def count_files_in_dir(directory):
    if not os.path.exists(directory):
        return 0
    count = 0
    for root, dirs, files in os.walk(directory):
        count += len(files)
    return count

dirs = ["../text_files/train", "../text_files/test", "../text_files/valid"]
for d in dirs:
    print(f"{d}: {count_files_in_dir(d)} files")

../text_files/train: 5264 files
../text_files/test: 2256 files
../text_files/valid: 1880 files


In [5]:
# ============================================
# DATA LOADING FUNCTIONS
# ============================================

def load_text_data(folder):
    """Load text files and return texts, labels, and filenames"""
    texts, labels, filenames = [], [], []
    
    # Check if folder exists
    if not os.path.exists(folder):
        print(f"Warning: {folder} does not exist")
        return [], np.array([]), []
    
    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)
        
        if not os.path.exists(subfolder):
            print(f"Warning: {subfolder} does not exist")
            continue
            
        for file in sorted(os.listdir(subfolder)):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)
                filenames.append((label_name, file))
    
    return texts, np.array(labels), filenames


def load_image_paths(image_folder, filenames):
    """Load image paths corresponding to text files"""
    image_paths = []
    
    for label_name, file in filenames:
        img_name = file.replace(".txt", ".png")
        img_path = os.path.join(image_folder, label_name, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Missing image: {img_path}")
            # Try alternative extensions
            for ext in ['.jpg', '.jpeg']:
                alt_path = img_path.replace('.png', ext)
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break
            else:
                raise FileNotFoundError(f"Missing image: {img_path}")
        
        image_paths.append(img_path)
    
    return image_paths

In [6]:
from transformers import T5EncoderModel, AutoTokenizer

# Use CodeT5+ encoder
MODEL_NAME = "Salesforce/codet5p-220m"
NUM_LABELS = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# For the image processor we keep the same ViT processor
processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Hybrid model using CodeT5+ text features, DeiT/ViT image features, and optional tabular features.
class HybridModel(nn.Module):
    def __init__(self, use_cb, use_vit, extra_dim, tokenizer=None):
        super().__init__()
        self.use_cb = use_cb
        self.use_vit = use_vit
        self.extra_dim = extra_dim
        self.text_embedding_size = None

        if use_cb:
            self.codet5p = T5EncoderModel.from_pretrained(MODEL_NAME)
            if tokenizer is not None and len(tokenizer) != self.codet5p.get_input_embeddings().num_embeddings:
                self.codet5p.resize_token_embeddings(len(tokenizer))
            self.text_embedding_size = self.codet5p.get_input_embeddings().num_embeddings
            self.cb_dim = self.codet5p.config.d_model
        else:
            self.cb_dim = 0

        if use_vit:
            self.vit = ViTModel.from_pretrained("facebook/deit-base-patch16-224")
            self.vit_dim = self.vit.config.hidden_size
        else:
            self.vit_dim = 0

        input_dim = self.cb_dim + self.vit_dim + extra_dim
        if input_dim == 0:
            raise ValueError("At least one real feature source must be selected.")

        print("cb_dim:", self.cb_dim)
        print("vit_dim:", self.vit_dim)
        print("extra_dim:", extra_dim)

        self.norm = nn.LayerNorm(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_LABELS)
        )

    def forward(self, batch):
        feats = []

        if self.use_cb:
            input_ids = batch["input_ids"]
            if self.text_embedding_size is not None:
                min_token_id = int(input_ids.min().detach().cpu())
                max_token_id = int(input_ids.max().detach().cpu())
                if min_token_id < 0 or max_token_id >= self.text_embedding_size:
                    raise ValueError(
                        f"Token id out of range for CodeT5+ embeddings: "
                        f"min={min_token_id}, max={max_token_id}, embedding_size={self.text_embedding_size}. "
                        "Reload the matching tokenizer/model or resize token embeddings."
                    )

            out = self.codet5p(
                input_ids=input_ids,
                attention_mask=batch["attention_mask"],
                return_dict=True,
            )

            # Mean-pool valid tokens instead of using position 0 as a CLS token.
            mask = batch["attention_mask"].unsqueeze(-1).to(out.last_hidden_state.dtype)
            pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            feats.append(pooled)

        if self.use_vit:
            out = self.vit(pixel_values=batch["pixel_values"], return_dict=True)
            feats.append(out.last_hidden_state[:, 0, :])

        if batch["extra"].shape[1] > 0:
            feats.append(batch["extra"])

        x = torch.cat(feats, dim=1)
        x = self.norm(x)
        return self.classifier(x)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [7]:

print("Vocab size:", tokenizer.vocab_size)

Vocab size: 32100


In [8]:
# ============================================
# DATASET CLASS
# ============================================

class MultiModalDataset(Dataset):
    def __init__(self, texts, images, labels, extra, tokenizer, processor, max_length=256):
        self.texts = texts
        self.images = images
        self.labels = np.asarray(labels, dtype=np.int64)
        self.extra = np.asarray(extra, dtype=np.float32) if extra is not None else None
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length

        if len(self.texts) != len(self.images) or len(self.texts) != len(self.labels):
            raise ValueError(
                f"Mismatched dataset lengths: texts={len(self.texts)}, "
                f"images={len(self.images)}, labels={len(self.labels)}"
            )
        if self.extra is not None and len(self.extra) not in (0, len(self.texts)):
            raise ValueError(f"Extra feature length {len(self.extra)} does not match texts {len(self.texts)}")
        if len(self.labels) and (self.labels.min() < 0 or self.labels.max() >= NUM_LABELS):
            raise ValueError(f"Labels must be in [0, {NUM_LABELS - 1}], got {np.unique(self.labels)}")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        with Image.open(self.images[idx]) as img:
            img = img.convert("RGB")

            text_enc = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                add_special_tokens=True,
                return_tensors="pt"
            )
            img_enc = self.processor(images=img, return_tensors="pt")

        if self.extra is not None and self.extra.shape[1] > 0:
            extra = torch.tensor(self.extra[idx], dtype=torch.float32)
        else:
            extra = torch.zeros(0, dtype=torch.float32)

        return {
            "input_ids": text_enc["input_ids"].squeeze(0).long(),
            "attention_mask": text_enc["attention_mask"].squeeze(0).long(),
            "pixel_values": img_enc["pixel_values"].squeeze(0).float(),
            "extra": extra,
            "label": torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


In [9]:
# ============================================
# FEATURE EXTRACTION
# ============================================

def extract_stylometric_features(code: str):
    """Extract 4 stylometric features"""
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    
    return np.array([
        avg_line_length,
        len(lines),
        len(code.split()),
        len(code)
    ], dtype=np.float32)


def compute_stylo(texts):
    """Compute stylometric features for all texts"""
    if len(texts) == 0:
        return np.array([]).reshape(0, 4)
    return np.array([extract_stylometric_features(t) for t in texts])


def load_metric_features(path, expected_labels, split_name):
    """Load precomputed metrics and verify they match the current split order."""
    data = np.load(path)
    if "metrics" not in data or "labels" not in data:
        raise KeyError(f"{path} must contain 'metrics' and 'labels' arrays")

    metrics = data["metrics"].astype(np.float32)
    metric_labels = data["labels"].astype(np.int64)
    expected_labels = np.asarray(expected_labels, dtype=np.int64)

    if metrics.shape[0] != len(expected_labels):
        raise ValueError(
            f"{split_name} metrics row count {metrics.shape[0]} does not match "
            f"loaded labels {len(expected_labels)}"
        )
    if not np.array_equal(metric_labels, expected_labels):
        raise ValueError(
            f"{split_name} metric labels do not match the current text loading order. "
            "Regenerate metrics with the same split folders/order."
        )

    return metrics

In [10]:
# Load splits
print("Loading train/val/test data from new folders...")
train_texts, train_labels, train_files = load_text_data("../text_files/train")
val_texts, val_labels, val_files = load_text_data("../text_files/valid")
test_texts, test_labels, test_files = load_text_data("../text_files/test")

print("TRAIN LABELS:", np.unique(train_labels), train_labels.min(), train_labels.max())
print("VAL LABELS:", np.unique(val_labels), val_labels.min(), val_labels.max())
print("TEST LABELS:", np.unique(test_labels), test_labels.min(), test_labels.max())

assert train_labels.min() >= 0 and train_labels.max() < NUM_LABELS
assert val_labels.min() >= 0 and val_labels.max() < NUM_LABELS
assert test_labels.min() >= 0 and test_labels.max() < NUM_LABELS

# Load corresponding images
train_imgs = load_image_paths("../snapshots/train", train_files)
val_imgs = load_image_paths("../snapshots/valid", val_files)
test_imgs = load_image_paths("../snapshots/test", test_files)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

# TF-IDF: fit on train, transform all
vectorizer = TfidfVectorizer(max_features=500)
train_tfidf = vectorizer.fit_transform(train_texts).toarray().astype(np.float32)
val_tfidf = vectorizer.transform(val_texts).toarray().astype(np.float32)
test_tfidf = vectorizer.transform(test_texts).toarray().astype(np.float32)

# Stylometric features (computed on the fly)
train_stylo = compute_stylo(train_texts)
val_stylo = compute_stylo(val_texts)
test_stylo = compute_stylo(test_texts)

# Precomputed CodeBERT/GLTR-style metrics
train_metrics = load_metric_features("../metrics/metrics_train.npz", train_labels, "train")
val_metrics = load_metric_features("../metrics/metrics_valid.npz", val_labels, "valid")
test_metrics = load_metric_features("../metrics/metrics_test.npz", test_labels, "test")

features = ["codet5p", "vit", "stylometric", "tfidf", "metrics"]


Loading train/val/test data from new folders...
TRAIN LABELS: [0 1] 0 1
VAL LABELS: [0 1] 0 1
TEST LABELS: [0 1] 0 1
Train: 5264, Val: 1880, Test: 2256


In [11]:
# ============================================
# TRAINING AND EVALUATION WITH VALIDATION
# ============================================

def format_duration(seconds):
    seconds = int(max(seconds, 0))
    hours, rem = divmod(seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def move_batch_to_device(batch, device):
    return {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}


def validate_batch(batch):
    labels = batch["label"]
    if labels.dtype != torch.long:
        raise TypeError(f"CrossEntropyLoss expects torch.long labels, got {labels.dtype}")
    min_label = int(labels.min().detach().cpu())
    max_label = int(labels.max().detach().cpu())
    if min_label < 0 or max_label >= NUM_LABELS:
        raise ValueError(f"Labels must be in [0, {NUM_LABELS - 1}], got min={min_label}, max={max_label}")


def train_and_eval(combo,
                   train_texts, train_labels, train_imgs, train_tfidf, train_metrics,
                   val_texts, val_labels, val_imgs, val_tfidf, val_metrics,
                   test_texts, test_labels, test_imgs, test_tfidf, test_metrics,
                   device, tokenizer, processor,
                   max_epochs=3, patience=2):

    combo_start = time.time()
    device = torch.device(device)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    use_cb = "codebert" in combo or "codet5p" in combo
    use_vit = "vit" in combo

    # ---- Build training extra features ----
    extra_list = []
    if "stylometric" in combo and len(train_texts) > 0:
        extra_list.append(compute_stylo(train_texts))
    if "tfidf" in combo and train_tfidf is not None:
        extra_list.append(train_tfidf)
    if "metrics" in combo and train_metrics is not None:
        extra_list.append(train_metrics)

    scaler = StandardScaler()
    if extra_list:
        extra = np.concatenate(extra_list, axis=1).astype(np.float32)
        extra = scaler.fit_transform(extra).astype(np.float32)
    else:
        extra = np.zeros((len(train_texts), 0), dtype=np.float32)

    # ---- Build validation extra features ----
    val_extra_list = []
    if "stylometric" in combo and len(val_texts) > 0:
        val_extra_list.append(compute_stylo(val_texts))
    if "tfidf" in combo and val_tfidf is not None:
        val_extra_list.append(val_tfidf)
    if "metrics" in combo and val_metrics is not None:
        val_extra_list.append(val_metrics)

    if val_extra_list:
        val_extra = np.concatenate(val_extra_list, axis=1).astype(np.float32)
        val_extra = scaler.transform(val_extra).astype(np.float32)
    else:
        val_extra = np.zeros((len(val_texts), 0), dtype=np.float32)

    # ---- Build test extra features ----
    test_extra_list = []
    if "stylometric" in combo and len(test_texts) > 0:
        test_extra_list.append(compute_stylo(test_texts))
    if "tfidf" in combo and test_tfidf is not None:
        test_extra_list.append(test_tfidf)
    if "metrics" in combo and test_metrics is not None:
        test_extra_list.append(test_metrics)

    if test_extra_list:
        test_extra = np.concatenate(test_extra_list, axis=1).astype(np.float32)
        test_extra = scaler.transform(test_extra).astype(np.float32)
    else:
        test_extra = np.zeros((len(test_texts), 0), dtype=np.float32)

    # ---- Datasets and loaders ----
    train_dataset = MultiModalDataset(train_texts, train_imgs, train_labels, extra, tokenizer, processor)
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=(device.type == "cuda"))

    val_dataset = MultiModalDataset(val_texts, val_imgs, val_labels, val_extra, tokenizer, processor)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, pin_memory=(device.type == "cuda"))

    test_dataset = MultiModalDataset(test_texts, test_imgs, test_labels, test_extra, tokenizer, processor)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, pin_memory=(device.type == "cuda"))

    # ---- Model and optimizer ----
    print("extra_dim:", extra.shape[1])
    model = HybridModel(use_cb, use_vit, extra.shape[1], tokenizer=tokenizer).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()

    # ---- Training with validation and early stopping ----
    best_val_acc = -1.0
    best_model_state = None
    epochs_no_improve = 0
    combo_name = "+".join(combo)
    total_steps = max_epochs * (len(train_loader) + len(val_loader)) + len(test_loader)

    with tqdm(total=total_steps, desc=f"{combo_name} train/val/test", unit="batch", leave=True) as combo_pbar:
        for epoch in range(max_epochs):
            model.train()
            total_loss = 0.0
            combo_pbar.set_postfix_str(f"epoch {epoch + 1}/{max_epochs} train")
            for batch in train_loader:
                batch = move_batch_to_device(batch, device)
                validate_batch(batch)

                optimizer.zero_grad(set_to_none=True)
                outputs = model(batch)
                loss = loss_fn(outputs, batch["label"])
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                combo_pbar.update(1)

            avg_loss = total_loss / max(len(train_loader), 1)

            model.eval()
            val_preds, val_truth = [], []
            combo_pbar.set_postfix_str(f"epoch {epoch + 1}/{max_epochs} validate")
            with torch.no_grad():
                for batch in val_loader:
                    batch = move_batch_to_device(batch, device)
                    validate_batch(batch)
                    outputs = model(batch)
                    val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
                    val_truth.extend(batch["label"].cpu().numpy())
                    combo_pbar.update(1)

            val_acc = accuracy_score(val_truth, val_preds)
            elapsed = time.time() - combo_start
            completed_steps = combo_pbar.n
            eta = (elapsed / completed_steps) * (total_steps - completed_steps) if completed_steps else 0
            print(
                f"  Epoch {epoch+1}/{max_epochs} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.4f} "
                f"- Elapsed: {format_duration(elapsed)} - ETA: {format_duration(eta)}"
            )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    skipped_steps = (max_epochs - epoch - 1) * (len(train_loader) + len(val_loader))
                    if skipped_steps:
                        combo_pbar.update(skipped_steps)
                    print(f"  Early stopping after epoch {epoch+1}")
                    break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        model.to(device)

        # ---- Final evaluation on test set ----
        model.eval()
        preds, ys = [], []
        combo_pbar.set_postfix_str("test")
        with torch.no_grad():
            for batch in test_loader:
                batch = move_batch_to_device(batch, device)
                validate_batch(batch)
                outputs = model(batch)
                preds.extend(outputs.argmax(dim=1).cpu().numpy())
                ys.extend(batch["label"].cpu().numpy())
                combo_pbar.update(1)

    test_acc = accuracy_score(ys, preds)
    total_elapsed = time.time() - combo_start
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Combination finished in {format_duration(total_elapsed)}")

    del model, optimizer, train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return test_acc


In [12]:
# ============================================
# MAIN EXECUTION
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and processor
print("Loading tokenizer and processor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")


Using device: cuda
Loading tokenizer and processor...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [13]:
# Load training data
print("Loading training data...")
train_texts, train_labels, train_files = load_text_data("../text_files/train")
if len(train_texts) == 0:
    print("Error: No training data found!")
    exit(1)
print(f"Loaded {len(train_texts)} training samples")
train_imgs = load_image_paths("../snapshots/train", train_files)
print(f"Loaded {len(train_imgs)} training images")

Loading training data...
Loaded 5264 training samples
Loaded 5264 training images


In [14]:
# Load validation data
print("Loading validation data...")
val_texts, val_labels, val_files = load_text_data("../text_files/valid")
if len(val_texts) == 0:
    print("Warning: No validation data found! Exiting.")
    exit(1)
else:
    print(f"Loaded {len(val_texts)} validation samples")
    val_imgs = load_image_paths("../snapshots/valid", val_files)
    print(f"Loaded {len(val_imgs)} validation images")

Loading validation data...
Loaded 1880 validation samples
Loaded 1880 validation images


In [15]:
# Load test data (single set)
print("Loading test data...")
test_texts, test_labels, test_files = load_text_data("../text_files/test")
if len(test_texts) == 0:
    print("Error: No test data found!")
    exit(1)
print(f"Loaded {len(test_texts)} test samples")
test_imgs = load_image_paths("../snapshots/test", test_files)
print(f"Loaded {len(test_imgs)} test images")

Loading test data...
Loaded 2256 test samples
Loaded 2256 test images


In [16]:
# Compute TF-IDF features
print("Computing TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=500)
train_tfidf = vectorizer.fit_transform(train_texts).toarray().astype(np.float32)
val_tfidf = vectorizer.transform(val_texts).toarray().astype(np.float32)
test_tfidf = vectorizer.transform(test_texts).toarray().astype(np.float32)

print("Loading precomputed metrics...")
train_metrics = load_metric_features("../metrics/metrics_train.npz", train_labels, "train")
val_metrics = load_metric_features("../metrics/metrics_valid.npz", val_labels, "valid")
test_metrics = load_metric_features("../metrics/metrics_test.npz", test_labels, "test")
print(f"Metrics: train {train_metrics.shape}, valid {val_metrics.shape}, test {test_metrics.shape}")


Computing TF-IDF features...
Loading precomputed metrics...
Metrics: train (5264, 5), valid (1880, 5), test (2256, 5)


In [17]:
# Feature combinations
features = ["codet5p", "vit", "stylometric", "tfidf", "metrics"]

best_acc = 0
best_combo = None


In [18]:
# Try all combinations
all_combos = [combo for r in range(1, len(features) + 1) for combo in itertools.combinations(features, r)]
sweep_start = time.time()

with tqdm(total=len(all_combos), desc="All feature combinations", unit="combo", leave=True) as sweep_pbar:
    for combo_index, combo in enumerate(all_combos, start=1):
        combo_start = time.time()
        print(f"\n{'='*50}")
        print(f"Testing combination {combo_index}/{len(all_combos)}: {combo}")
        print(f"{'='*50}")
        try:
            acc = train_and_eval(
                combo,
                train_texts, train_labels, train_imgs, train_tfidf, train_metrics,
                val_texts, val_labels, val_imgs, val_tfidf, val_metrics,
                test_texts, test_labels, test_imgs, test_tfidf, test_metrics,
                device, tokenizer, processor
            )
            print(f"\n>>> Test Accuracy for {combo}: {acc:.4f} <<<")
            print(f">>> Combination runtime: {format_duration(time.time() - combo_start)} <<<\n")
            if acc > best_acc:
                best_acc = acc
                best_combo = combo
        except RuntimeError as e:
            if device.type == "cuda" and "CUDA out of memory" in str(e):
                torch.cuda.empty_cache()
            print(f"Error with combination {combo}: {e}")
        finally:
            sweep_pbar.update(1)
            elapsed = time.time() - sweep_start
            remaining = len(all_combos) - sweep_pbar.n
            eta = (elapsed / sweep_pbar.n) * remaining if sweep_pbar.n else 0
            sweep_pbar.set_postfix_str(f"elapsed {format_duration(elapsed)}, eta {format_duration(eta)}")

print("\n" + "="*50)
print(f"BEST COMBINATION: {best_combo}")
print(f"BEST TEST ACCURACY: {best_acc:.4f}")
print(f"TOTAL SWEEP TIME: {format_duration(time.time() - sweep_start)}")
print("="*50)


All feature combinations:   0%|          | 0/31 [00:00<?, ?combo/s]


Testing combination 1/31: ('codet5p',)
extra_dim: 0
cb_dim: 768
vit_dim: 0
extra_dim: 0


codet5p train/val/test:   0%|          | 0/5922 [00:00<?, ?batch/s]

  Epoch 1/3 - Loss: 0.5300 - Val Acc: 0.8186 - Elapsed: 3m 49s - ETA: 8m 52s
  Epoch 2/3 - Loss: 0.3211 - Val Acc: 0.8335 - Elapsed: 9m 49s - ETA: 6m 27s
  Epoch 3/3 - Loss: 0.2117 - Val Acc: 0.8452 - Elapsed: 16m 07s - ETA: 1m 41s
  Test Accuracy: 0.8404
  Combination finished in 16m 57s

>>> Test Accuracy for ('codet5p',): 0.8404 <<<
>>> Combination runtime: 16m 57s <<<


Testing combination 2/31: ('vit',)
extra_dim: 0


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

  Epoch 1/3 - Loss: 0.6752 - Val Acc: 0.6415 - Elapsed: 1m 35s - ETA: 3m 41s
  Epoch 2/3 - Loss: 0.5711 - Val Acc: 0.7112 - Elapsed: 3m 10s - ETA: 2m 05s
  Epoch 3/3 - Loss: 0.5061 - Val Acc: 0.7293 - Elapsed: 4m 45s - ETA: 30s
  Test Accuracy: 0.7026
  Combination finished in 4m 57s

>>> Test Accuracy for ('vit', 'stylometric', 'tfidf', 'metrics'): 0.7026 <<<
>>> Combination runtime: 4m 57s <<<


Testing combination 31/31: ('codet5p', 'vit', 'stylometric', 'tfidf', 'metrics')
extra_dim: 509


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cb_dim: 768
vit_dim: 768
extra_dim: 509


codet5p+vit+stylometric+tfidf+metrics train/val/test:   0%|          | 0/5922 [00:00<?, ?batch/s]

  Epoch 1/3 - Loss: 0.5793 - Val Acc: 0.7883 - Elapsed: 3m 36s - ETA: 8m 20s
  Epoch 2/3 - Loss: 0.3692 - Val Acc: 0.8234 - Elapsed: 7m 00s - ETA: 4m 36s
  Epoch 3/3 - Loss: 0.2444 - Val Acc: 0.8378 - Elapsed: 10m 25s - ETA: 1m 05s
  Test Accuracy: 0.8147
  Combination finished in 10m 46s

>>> Test Accuracy for ('codet5p', 'vit', 'stylometric', 'tfidf', 'metrics'): 0.8147 <<<
>>> Combination runtime: 10m 46s <<<


BEST COMBINATION: ('codet5p', 'metrics')
BEST TEST ACCURACY: 0.8422
TOTAL SWEEP TIME: 5h 49m 53s
